**Molecular physical pharmacy, 3FC003**

**Molecular dynamics exercise**

# High-throughput peptide self-assembly

**Introduction**

This exercise will introduce you to combinatorial screening for peptide self-assembly using the Martini force field for proteins. The process is automated using a number of bash scripts, Gromacs tools and scripting capabilities within the visual molecular dynamics (VMD) program. After the simulations, visual inspection is done using VMD and analysis of the assembled structures is done using Gromacs tools.

NOTE that the exercise is written for versions 5.1 or 2016/2018 of Gromacs and will show errors when used with earlier versions

# Important:

Before start with this lab click in the edit menu and select clear all output

# 1. Setting up the computing environment

In [1]:
# Installing necessary packages
!pip install py3Dmol

In [2]:
# Cloning the course repository
!git clone https://github.com/computationalpharmaceutics/3FC003.git

Cloning into '3FC003'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 13 (delta 3), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 2.61 MiB | 1.98 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [3]:
# Installing GROMACS with CPU support only
!apt install gromacs &> /dev/null

In [4]:
# Confirm it's installed
!gmx -version

              :-) GROMACS - gmx, 2023.3-Ubuntu_2023.3_1ubuntu3 (-:

Executable:   /usr/bin/gmx
Data prefix:  /usr
Working dir:  /content
Command line:
  gmx -version

GROMACS version:    2023.3-Ubuntu_2023.3_1ubuntu3
Precision:          mixed
Memory model:       64 bit
MPI library:        thread_mpi
OpenMP support:     enabled (GMX_OPENMP_MAX_THREADS = 128)
GPU support:        disabled
SIMD instructions:  SSE4.1
CPU FFT library:    fftw-3.3.10-sse2-avx
GPU FFT library:    none
Multi-GPU FFT:      none
RDTSCP usage:       enabled
TNG support:        enabled
Hwloc support:      hwloc-2.8.0
Tracing support:    disabled
C compiler:         /usr/bin/cc GNU 13.2.0
C compiler flags:   -fexcess-precision=fast -funroll-all-loops -msse4.1 -Wno-missing-field-initializers -O3 -DNDEBUG
C++ compiler:       /usr/bin/c++ GNU 13.2.0
C++ compiler flags: -fexcess-precision=fast -funroll-all-loops -msse4.1 -Wno-missing-field-initializers -Wno-cast-function-type-strict SHELL:-fopenmp -O3 -DNDEBUG
BLAS libr

In [5]:
# Looking at the initial files
import os
os.chdir('/content/3FC003/HT_peptide_self_assembly')
!ls -l

total 3732
-rw-r--r-- 1 root root  151381 Sep 12 19:14 example.ipynb
-rw-r--r-- 1 root root 3591536 Sep 12 19:14 example_peptide_aggregation_CPU.ipynb
-rwxr-xr-x 1 root root    1262 Sep 12 19:14 inertia_trip.txt
-rw-r--r-- 1 root root   65477 Sep 12 19:14 LAB_Molecular_dynamics_peptide_self_assembly_CPU.ipynb
-rwxr-xr-x 1 root root     134 Sep 12 19:14 Peptide_assembly.tgz
-rwxr-xr-x 1 root root     381 Sep 12 19:14 sasa_trip.txt


In [6]:
# Setting the path to GROMACS
import os
os.environ["PATH"] += ":/usr/local/gromacs/bin"

# Looking for the working directory
!pwd

/content/3FC003/HT_peptide_self_assembly


In [7]:
# Changing the working directory to

import os
os.chdir('/content/3FC003/HT_peptide_self_assembly')
!pwd

/content/3FC003/HT_peptide_self_assembly


# 1 Introduction

This exercise will introduce you to combinatorial screening for peptide self-assembly using the Martini force field for proteins. The process is automated using a number of bash scripts, Gromacs tools and scripting capabilities within the visual molecular dynamics (VMD) program. After the simulations, visual inspection is done using VMD and analysis of the assembled structures is done using Gromacs tools. NOTE that the exercise is written for versions 5.1 or 2016/2018 of Gromacs and will show errors when used with earlier versions.

- The exercise discusses the self-assembly of short peptides as an example system.
- The material for the exercise is located in the file **Peptide_assembly.tgz**.
- Unpack the directory tree (it expands to a directory called Peptide_assembly_GMX5-2016):

In [8]:
# Unpack the course material
!apt-get -qq update && apt-get -qq install -y git-lfs
!git lfs install --local
!git lfs pull
!tar -xf Peptide_assembly.tgz

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package git-lfs.
(Reading database ... 150665 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Updated Git hooks.
Git LFS initialized.


The material is organized in a directory tree that is numbered by the subsections of this tutorial:

 1_Background
  
 2_Creating_coordinates
  
 3_Coarse-graining

 4_Running_simulations

 5_Analysis

Each directory tree contains the files required for the tutorial. The results of a successful execution of the tutorial are also provided; this enables you to check your work and also to start anywhere if you run into some problem we cannot easily solve together. For example, if you want to jump in at step 4, you can enter the directory 3_Done and continue there in the directory 4_Running_simulations. Be advised that you do so at your own peril...

The main idea of this exercise is to show you, and practice, the commands to get through the different steps for a single peptide. However, in the files provided, you will also find several scripts that can help you perform the operations automatically, and set up a high throughput assay.

Finally, this exercise is based on a tutorial available on the martini website (cgmartini.nl), which is gratefully acknowledged.

**1. Background**

Molecular self-assembly of oligopeptides into nanostructures holds much promise for a range of potential applications in biomedicine, food science, cosmetics and nanotechnology. This class of materials is highly versatile because of the combinatorial complexity achieved by combining 20 amino acids into peptide building blocks with a wide range of chemical functionality. The use of very short peptides is especially attractive, enhancing opportunities for rational design combined with robustness, scalability and cost reduction.

Two main challenges are currently limiting the expansion of this field. Most examples of short peptides contain only hydrophobic amino acids. This is no surprise as hydrophobic interactions dominate self-assembly in water, but it also limits their aqueous solubility and restricts potential applications. Secondly, in spite of two decades of intensive research since the first examples of short self-assembling peptides, most examples have been either discovered by serendipity or by mapping onto known sequence design rules from biological systems. Using Martini, the self-assembly of thousands of peptides can be tested in silico, before spending time and resources in the lab. Experimental validation has shown that Martini not only accurately represents the level of aggregation between peptides, but also informs on the supramolecular structure of the nanosized assemblies.

In the field of peptide nanomaterials, it is common practice to ‘protect’ the N- and/or Cterminus of a peptide to introduce specific interactions and remove charge-charge repulsions. Examples include acetyl, Fmoc, naphthalene, pyrene and t-Boc functional groups at the N-terminus, or amide and ester groups at the C-terminus.

The directory 1_Background contains some references and further reading for those that are interested, but reading this is optional.

In this exercise we will apply a combinatorial screening protocol to tripeptides, specifically those discovered by Ray et al. They found that placing tyrosine residues at both the N- and the C-terminus of a tripeptide drives the system to crystallize into hollow nanotubes with a 5 Å inner diameter. They showed this works for Boc-Tyr-X-Tyr-OMe peptides, where X = Val, Ile, and that mutating either of the tyrosine residues prevents nanotube formation. This begs the question if peptides with other middle residues will maintain the nanotube conformation or will change their morphology. Additionally, the nanotubes were created by crystallization from a water/methanol mixture, while for realistic applications, the stability of the nanotubes in an aqueous environment should be tested.

In [10]:
import os

# Define the source path of the file to be downloaded
source_path = '/content/3FC003/HT_peptide_self_assembly/Peptide_assembly_GMX5-2016'

# Define the destination path (current working directory)
destination_path = '.'

# Download the file (assuming it's a directory or a large file that needs to be moved/copied)
# Since it's a directory, I'll use a copy command or simply indicate it's already extracted by tar -xf
# However, the download prompt usually means a file *from* this location, not *to* it.
# Given the context of previous cells (tar -xf Peptide_assembly.tgz), the directory 'Peptide_assembly_GMX5-2016' should already exist.
# I will confirm its presence and list its content.

if os.path.exists(source_path):
    print(f"The directory '{source_path}' already exists. Listing its contents:")
    !ls -l {source_path}
else:
    print(f"The directory '{source_path}' does not exist. It should have been extracted by `tar -xf Peptide_assembly.tgz`.")
    print("Please ensure the previous cell for unpacking the course material was run successfully.")


The directory '/content/3FC003/HT_peptide_self_assembly/Peptide_assembly_GMX5-2016' already exists. Listing its contents:
total 52
drwxr-xr-x 2 23068 150  4096 Jun  5  2016 1_Background
drwxr-xr-x 2 23068 150  4096 Aug  1  2017 2_Creating_coordinates
drwxr-xr-x 2 23068 150  4096 Aug  2  2017 2_Done
drwxr-xr-x 2 23068 150  4096 Aug  1  2017 3_Coarse-graining
drwxr-xr-x 2 23068 150  4096 Aug  2  2017 3_Done
drwxr-xr-x 2 23068 150 12288 Aug  2  2017 4_Done
drwxr-xr-x 2 23068 150  4096 Aug  2  2017 4_Running_simulations
drwxr-xr-x 2 23068 150  4096 Aug  2  2017 5_Analysis
drwxr-xr-x 2 23068 150 12288 Aug  2  2017 5_Done


In [11]:
# Create a zip archive of the directory for easy download
import shutil
import os

source_dir = '/content/3FC003/HT_peptide_self_assembly/Peptide_assembly_GMX5-2016'
output_filename = 'Peptide_assembly_GMX5-2016'

if os.path.exists(source_dir):
    shutil.make_archive(output_filename, 'zip', source_dir)
    print(f"Successfully created '{output_filename}.zip' in the current working directory. You can find it in the Colab file browser (folder icon on the left) and download it from there.")
else:
    print(f"Error: The directory '{source_dir}' does not exist. Cannot create zip archive.")

Successfully created 'Peptide_assembly_GMX5-2016.zip' in the current working directory. You can find it in the Colab file browser (folder icon on the left) and download it from there.
